In [ ]:
import h5py
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
%matplotlib inline

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

## Omission responses by cluster, cell type, and layer

### By cluster

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def plot_omission_psth_by_group(values, filter_kwargs_fn, title_fn,
                                inset_loc_fn=None, ylim_fn=None):
    fig, axes = plt.subplots(len(values), 1)
    fig.set_size_inches(7, 2.5 * len(values))
    session_list = list(active_tensor.keys())
    time = np.arange(-750, 750, 1)

    for ax, value in zip(np.atleast_1d(axes), values):
        unit_ids = vbn_utils.get_unit_ids(
            units, 'VISall', clustering='new', **filter_kwargs_fn(value)
        )

        omission_responses, _ = vbn_utils.unit_averaged_psth(
            active_tensor_file, stim_table, session_list, unit_ids,
            'omitted', baseline_length=750, resp_window_length=750,
        )

        all_responses = np.array([
            exponential_convolve(ors, tau=3, symmetrical=True)
            for oresps in omission_responses for ors in oresps
        ])

        ax.set_title(title_fn(value), fontsize=16, pad=12, loc='right')
        vbn_utils.mean_sem_plot(all_responses * 1000, ax, time, color='k')

        if ylim_fn is not None and ylim_fn(value) is not None:
            ax.set_ylim(*ylim_fn(value))

        rect_h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05
        ax.add_patch(Rectangle((-750, ax.get_ylim()[1]), 250, rect_h,
                               color='gray', alpha=1, clip_on=False))
        ax.add_patch(Rectangle((0, ax.get_ylim()[1]), 250, rect_h,
                               edgecolor='gray', facecolor='none',
                               linewidth=2, clip_on=False))
        baseline = all_responses.mean(axis=0)[0] * 1000
        ax.axhline(baseline, color='k', linestyle='--', linewidth=1)
        ax.set_xlim(-750, 750)
        vbn_utils.formatFigure(fig, ax)

        loc, borderpad = inset_loc_fn(value) if inset_loc_fn else ('upper right', 0.5)
        inset_ax = inset_axes(ax, width="20%", height="40%", loc=loc, borderpad=borderpad)
        vbn_utils.mean_sem_plot(all_responses[:, 600:1100] * 1000, inset_ax,
                                time[600:1100], color='k')
        inset_ax.axhline(baseline, color='k', linestyle='--', linewidth=1)
        vbn_utils.formatFigure(fig, inset_ax)

    plt.tight_layout()
    return fig

In [ ]:
plot_omission_psth_by_group(
    values=np.arange(1, 9),
    filter_kwargs_fn=lambda c: dict(cell_types='all', clusters=c),
    title_fn=lambda c: f'Cluster {c}',
    inset_loc_fn=lambda c: ('lower right', 1.5) if c == 5 else ('upper right', 0.5),
)

### By cell type

In [ ]:
plot_omission_psth_by_group(
    values=['RS', 'FS', 'SST', 'VIP'],
    filter_kwargs_fn=lambda ct: dict(cell_types=ct, clusters='all', experience='all'),
    title_fn=lambda ct: f'Cell Type {ct}',
    inset_loc_fn=lambda ct: ('lower right', 1.5) if ct == 'VIP' else ('upper right', 0.5),
    ylim_fn=lambda ct: (-3, 15) if ct == 'VIP' else None,
)

### By cortical layer

In [ ]:
plot_omission_psth_by_group(
    values=['2/3', '4', '5', '6'],
    filter_kwargs_fn=lambda l: dict(cell_types='all', clusters='all', layers=l),
    title_fn=lambda l: f'Layer {l} RS',
)